## Access Animal Satellite Relay Tagging Ctd Realtime QC (Parquet)
This Jupyter notebook demonstrates how to access and plot animal_ctd_satellite_relay_tagging_location_qc_realtime_qc data, available as a [Parquet](https://parquet.apache.org) dataset stored on S3.

🔗 More information about the dataset is available [in the AODN metadata catalogue](https://catalogue-imos.aodn.org.au/geonetwork/srv/eng/catalog.search#/metadata/b2548767-514f-4a31-b65e-36bb894382d5).

📌 The source of truth for this notebook is maintained on [GitHub](https://github.com/aodn/aodn_cloud_optimised/tree/main/notebooks/animal_ctd_satellite_relay_tagging_location_qc_realtime_qc.ipynb).


In [ ]:
dataset_name = "animal_ctd_satellite_relay_tagging_location_qc_realtime_qc"

## Install/Update packages and Load common functions

In [ ]:
import os, requests, importlib.util

open('setup.py', 'w').write(requests.get('https://raw.githubusercontent.com/aodn/aodn_cloud_optimised/main/notebooks/setup.py').text)

spec = importlib.util.spec_from_file_location("setup", "setup.py")
setup = importlib.util.module_from_spec(spec)
spec.loader.exec_module(setup)

setup.install_requirements()
setup.load_dataquery()

In [ ]:
from DataQuery import GetAodn

# Understanding the Dataset

## Understanding Parquet Partitioning

Parquet files can be **partitioned** by one or more columns, which means the data is physically organised into folders based on the values in those columns. This is similar to how databases use indexes to optimise query performance.

Partitioning enables **faster filtering**: when you query data using a partitioned column, only the relevant subset of files needs to be read—improving performance significantly.

For example, if a dataset is partitioned by `"site_code"`, `"timestamp"`, and `"polygon"`, filtering on `"site_code"` allows the system to skip unrelated files entirely.

In this notebook, the `GetAodn` class includes built-in methods to efficiently filter data by **time** and **latitude/longitude** using the **timestamp** and **polygon** partitions. Other partitions can be used for filtering via the `scalar_filter`.

Any filtering on columns that are **not** partitioned can be significantly slower, as all files may need to be scanned. However, the `GetAodn` class provides a `scalar_filter` method that lets you apply these filters at load time—before the data is fully read—helping reduce the size of the resulting DataFrame.

Once the dataset is loaded, further filtering using Pandas is efficient and flexible.

See further below in the notebook for examples of how to filter the data effectively.

To view the actual partition columns for this dataset, run:


In [ ]:
aodn = GetAodn()
dname = f'{dataset_name}.parquet'
%time aodn_dataset = aodn.get_dataset(dname)

In [ ]:
aodn_dataset.dataset.partitioning.schema

## List unique partition values

In [ ]:
%%time
unique_partition_value = aodn_dataset.get_unique_partition_values('YOUR_PARTITION_KEY')
print(list(unique_partition_value)[0:2])  # showing a subset only

## Visualise Spatial Extent of the dataset
This section plots the polygons representing the areas where data is available. It helps to identify and create a bounding box around the regions containing data.

In [ ]:
aodn_dataset.plot_spatial_extent()

## Get Temporal Extent of the dataset

Similary to the spatial extent, we're retrieving the minimum and maximum timestamp partition values of the dataset. This is not necessarely accurately representative of the TIME values, as the timestamp partition can be yearly/monthly... but is here to give an idea

In [ ]:
%%time
aodn_dataset.get_temporal_extent()

## Read Metadata

For all Parquet datasets, we create a sidecar file named **_common_metadata** in the root of the dataset. This file contains both the dataset-level and variable-level attributes.  
The metadata can be retrieved below as a dictionary, and it will also be included in the pandas DataFrame when using the `get_data` method from the `GetAodn` class.

In [ ]:
metadata = aodn_dataset.get_metadata()
metadata

# Data Query and Plot

## Create a TIME and BoundingBox filter

This cell loads a subset of the dataset based on a time range and a spatial bounding box. The result is returned as a pandas DataFrame, and basic information about its structure is displayed.

In [ ]:
%%time
# NRT dataset, so we re just showing how to do a time filtering here taking the whole data
df = aodn_dataset.get_data(date_start=aodn_dataset.get_temporal_extent()[0].strftime('%Y-%m-%d'), 
                           date_end=aodn_dataset.get_temporal_extent()[1].strftime('%Y-%m-%d'),
                           )


df.info()

In [ ]:
## Download Subsetted Data as CSV

# This cell downloads the filtered dataset as a ZIP-compressed CSV file.  
# The CSV includes metadata at the top as commented lines, and a `FileLink` object is returned to allow downloading directly from the notebook.


#df.aodn.download_as_csv()

In [ ]:
df["ref"].unique()

## Create a subset

In [ ]:
df_subset = df[
    (df["ref"] == "ct189-596-25") & (df["end_date"] == "2026-04-01 04:00:00")
].sort_values(by="temp_dbar")

In [ ]:
df_subset

In [ ]:
import matplotlib.pyplot as plt

# Initialize a clean plot layout
fig, ax = plt.subplots(figsize=(5, 7))

# Plot Temperature against Pressure (dbar acts as depth)
ax.plot(
    df_subset["temp_vals"],
    df_subset["temp_dbar"],
    marker="o",
    linestyle="-",
    color="darkorange",
    linewidth=2,
    markersize=4,
)

# Invert the y-axis so greater depth/pressure is at the bottom
ax.invert_yaxis()

# Labeling with proper scientific notation
ax.set_xlabel("Temperature ($^\circ$C)")
ax.set_ylabel("Pressure ($\text{dbar}$)")
ax.set_title(
    f"Animal Profile: {df_subset['ref'].iloc[0]}\n{df_subset['end_date'].iloc[0]}"
)
ax.grid(True, linestyle=":", alpha=0.6)

plt.tight_layout()

# --- Quick Debug Block ---
print("--- Profile Plot Debug Summary ---")
print(f"Data points plotted: {len(df_subset)}")
print(
    f"Depth range: {df_subset['temp_dbar'].min()} to {df_subset['temp_dbar'].max()} dbar"
)
print(
    f"Temp range: {df_subset['temp_vals'].min()}$^\circ$C to {df_subset['temp_vals'].max()}$^\circ$C"
)
print("Missing temperature values:", df_subset["temp_vals"].isna().sum())
print("-----------------------------------")

## All animals tracks - Plot + Animation

In [ ]:
df = aodn_dataset.get_data()

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt

# 1. Setup the figure and map projection
# Using Mercator projection here; for polar/Antarctic tracks, ccrs.SouthPolarStereo() is also excellent.
plt.figure(figsize=(12, 9))
projection = ccrs.Mercator()
ax = plt.axes(projection=projection)

# 2. Add Land Contours and Ocean features
# '110m' is low resolution (fast), use '50m' or '10m' for highly detailed coastlines
ax.add_feature(cfeature.LAND, facecolor="lightgray", edgecolor="dimgray")
ax.add_feature(cfeature.OCEAN, facecolor="aliceblue", alpha=0.5)
ax.add_feature(cfeature.COASTLINE, linewidth=1.0, edgecolor="black")

# 3. Sort the entire dataframe chronologically
df_sorted = df.sort_values(by="end_date")

# 4. Group by the animal ID (ptt) and plot each track
for ptt_id, animal_df in df_sorted.groupby("ptt"):
    # IMPORTANT: Specify the coordinate system of your data (usually PlateCarree for standard Lat/Lon)
    (line,) = ax.plot(
        animal_df["ssm_lon"],
        animal_df["ssm_lat"],
        linestyle="-",
        linewidth=1.8,
        alpha=0.9,
        label=f"Animal PTT: {ptt_id}",
        transform=ccrs.PlateCarree(),  # Tells cartopy these are standard lat/lon degrees
    )

    # Add start (green) and end (red) markers
    if len(animal_df) > 0:
        ax.scatter(
            animal_df["ssm_lon"].iloc[0],
            animal_df["ssm_lat"].iloc[0],
            color="green",
            marker="^",
            s=50,
            zorder=3,
            transform=ccrs.PlateCarree(),
        )
        ax.scatter(
            animal_df["ssm_lon"].iloc[-1],
            animal_df["ssm_lat"].iloc[-1],
            color="red",
            marker="v",
            s=50,
            zorder=3,
            transform=ccrs.PlateCarree(),
        )

# 5. Highlight your target profile subset if it exists
if "df" in locals() and not df.empty:
    ax.scatter(
        df["ssm_lon"].iloc[0],
        df["ssm_lat"].iloc[0],
        color="magenta",
        marker="*",
        s=300,
        edgecolor="black",
        label="Target Profile",
        zorder=5,
        transform=ccrs.PlateCarree(),
    )

# 6. Dynamically zoom the map view slightly wider than the animal data bounds
pad = 2.0  # degrees of padding around the track bounds
lon_min, lon_max = df["ssm_lon"].min() - pad, df["ssm_lon"].max() + pad
lat_min, lat_max = df["ssm_lat"].min() - pad, df["ssm_lat"].max() + pad
ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())

# Add gridlines with map coordinates labeled
grid = ax.gridlines(draw_labels=True, linestyle="--", alpha=0.5, color="gray")
grid.top_labels = False
grid.right_labels = False

plt.title("Animal Trajectories over Land Contour", fontsize=14, pad=20)
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")

plt.tight_layout()

# --- Quick Debug Block ---
print("--- Land Contour Map Debug ---")
print(f"Map Extent Bounds: Lon({lon_min:.2f} to {lon_max:.2f}), Lat({lat_min:.2f} to {lat_max:.2f})")
print(f"Total rows rendered: {len(df_sorted)}")
print("------------------------------")


In [ ]:
import os
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.animation as animation
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import HTML

# Ensure inline plotting for Jupyter
%matplotlib inline

# Raise the animation embed limit to 100 Megabytes
plt.rcParams["animation.embed_limit"] = 100.0

# =========================================================================
# 1. DATA PREP 
# =========================================================================
df["end_date"] = pd.to_datetime(df["end_date"])
df = df.dropna(subset=["end_date", "ssm_lon", "ssm_lat"])
df["date_only"] = df["end_date"].dt.date
df_sorted = df.sort_values(by="end_date")
unique_days = sorted(df["date_only"].unique())
ptt_ids = df_sorted["ptt"].unique()

# Debug: Check if we have data
print(f"Total days to animate: {len(unique_days)}")

track_history = {day: {ptt: ([], []) for ptt in ptt_ids} for day in unique_days}
accumulated_data = {ptt: ([], []) for ptt in ptt_ids}
day_groups = df_sorted.groupby("date_only")

for day in unique_days:
    if day in day_groups.groups:
        day_df = day_groups.get_group(day)
        for ptt_id, animal_df in day_df.groupby("ptt"):
            lon_list, lat_list = accumulated_data[ptt_id]
            lon_list.extend(animal_df["ssm_lon"].tolist())
            lat_list.extend(animal_df["ssm_lat"].tolist())
            
    for ptt_id in ptt_ids:
        track_history[day][ptt_id] = (
            list(accumulated_data[ptt_id][0]),
            list(accumulated_data[ptt_id][1]),
        )

# =========================================================================
# 2. ANIMATION SETUP
# =========================================================================
pad = 2.0
lon_min, lon_max = df["ssm_lon"].min() - pad, df["ssm_lon"].max() + pad
lat_min, lat_max = df["ssm_lat"].min() - pad, df["ssm_lat"].max() + pad

# Create figure and axis with Cartopy projection
fig = plt.figure(figsize=(12, 9))
ax = plt.axes(projection=ccrs.PlateCarree())

# Add map features
ax.add_feature(cfeature.LAND, facecolor="lightgray", edgecolor="dimgray")
ax.add_feature(cfeature.OCEAN, facecolor="aliceblue")
ax.add_feature(cfeature.COASTLINE, linewidth=1.0, edgecolor="black")
ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
ax.grid(True, linestyle="--", alpha=0.3)

# Initialize lines
lines = {}
for ptt_id in ptt_ids:
    (line,) = ax.plot([], [], linestyle="-", linewidth=2.0, alpha=0.9, transform=ccrs.PlateCarree())
    lines[ptt_id] = line

title_text = ax.text(0.5, 1.02, "", transform=ax.transAxes, ha="center", va="bottom", fontsize=14)

def update(frame):
    current_day = unique_days[frame]
    for ptt_id, line in lines.items():
        lons, lats = track_history[current_day][ptt_id]
        line.set_data(lons, lats)
    
    title_text.set_text(f"Animal Tracks\nDate: {current_day.strftime('%Y-%m-%d')}")
    return list(lines.values()) + [title_text]

# Using blit=False is safer for complex projections like Cartopy
ani = animation.FuncAnimation(
    fig, update, frames=len(unique_days), interval=100, blit=False
)

# Render the animation
plt.close()
HTML(ani.to_jshtml())

In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt

target_ref = "ct189-596-25"
df_subset = df[df["ref"] == target_ref].sort_values(by="end_date")

# 1. Setup a grid layout: 2 rows, 2 columns
# Left column will hold the Profile (occupying both rows).
# Right column will split into Latitude (Top) and Longitude (Bottom).
fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(2, 2, width_ratios=[1.2, 1.0], wspace=0.3, hspace=0.3)

ax_profile = fig.add_subplot(gs[:, 0])  # Span both rows in col 0
ax_lat = fig.add_subplot(gs[0, 1])  # Row 0, col 1
ax_lon = fig.add_subplot(gs[1, 1], sharex=ax_lat)  # Row 1, col 1 (shares time axis)

# ==========================================
# PLOT 1: Temperature Profile (Left)
# ==========================================
sc = ax_profile.scatter(
    df_subset["end_date"],
    df_subset["temp_dbar"],
    c=df_subset["temp_vals"],
    marker="+",
    cmap="RdYlBu_r",
)
ax_profile.invert_yaxis()
ax_profile.set_title(
    f"Temperature Timeseries Profile\nRef: {target_ref}", fontsize=12, pad=10
)
ax_profile.set_ylabel("Depth (dbar)")
ax_profile.set_xlabel("Date")

# Add a colorbar dedicated to the profile plot
cbar = fig.colorbar(sc, ax=ax_profile, pad=0.03, aspect=30)
cbar.set_label("Temperature (°C)")


# ==========================================
# PLOT 2: Latitude Timeseries (Top Right)
# ==========================================
ax_lat.plot(
    df_subset["end_date"],
    df_subset["ssm_lat"],
    color="forestgreen",
    linestyle="-",
    marker=".",
    alpha=0.6,
)
ax_lat.set_title("Latitude Position over Time", fontsize=11)
ax_lat.set_ylabel("Latitude (°N/S)")
ax_lat.grid(True, linestyle="--", alpha=0.5)


# ==========================================
# PLOT 3: Longitude Timeseries (Bottom Right)
# ==========================================
ax_lon.plot(
    df_subset["end_date"],
    df_subset["ssm_lon"],
    color="darkorchid",
    linestyle="-",
    marker=".",
    alpha=0.6,
)
ax_lon.set_title("Longitude Position over Time", fontsize=11)
ax_lon.set_ylabel("Longitude (°E/W)")
ax_lon.set_xlabel("Date")
ax_lon.grid(True, linestyle="--", alpha=0.5)


# ==========================================
# FORMATTING & TIME ALIGNMENT
# ==========================================
# Rotate dates and format them cleanly across all axes
for ax_sub in [ax_profile, ax_lat, ax_lon]:
    ax_sub.tick_params(axis="x", labelrotation=45)
    # Ensures dates don't overlap awkwardly
    ax_sub.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))

# Hide the redundant top-right x-axis labels since it shares bounds with the bottom axis
plt.setp(ax_lat.get_xticklabels(), visible=False)
ax_lat.set_xlabel("")

plt.show()